# $\Delta H(p)$ calculations

In this notebook we compare the relative populations of gaps $w_{s,J}(\lambda)$ to samples of the populations
of gaps across the intervals of survival 
$$\Delta H(p_k) = (p_k^2, p_{k+1}^2]$$
We compare these actual counts to estimates made from the $w_{s,J}(\lambda)$.

For the models $w_{s,J}(p_k^\#)=w_{s,J}(\lambda)$ of relative populations, we use $p_0=37$.
The parameter
$$\lambda(p_k) = \prod_{p=41}^{p_k} \frac{p-J-2}{p-J-1} $$
So we create an array of values of $\lambda$ associated with the primes covered by the array <i>smallprimes[]</i>
for $p_0 = 37$.  The minimum value for $\lambda$ under this range is
$$ \lambda_{J=3}(p_k=1023094207) = 0.177736305553 $$
and
$$ \lambda_{J=5}(p_k=1023094207) = 0.175449847580 $$
Since our models start with $\lambda(37)=1$, there is an offset of 10 between the array of small primes 
and the array of the corresponding $\lambda$ 

This notebook prepares the data for comparing the actual populations of gaps between primes with the estimated populations
from the $w_{s,J}(\lambda)$.

This code includes 
* a function for identifying selected constellations among primes in long intervals among large primes.
* code for summarizing the populations of these gaps by interval of survival.



In [1]:
%reset -f

import numpy as np
import array
import pickle

import gc
import psutil
import sys

import itertools


In [2]:
# import the class AdmS for admissible constellations
# from a20s_class import AdmS
# import the dictionaries for the constellations of interest
from a20s_class import J3dict
from a20s_class import J5dict

from a20s_class import J3keyarr
from a20s_class import J5keyarr


## Loading the array of primes and the array for $\lambda$
We will be comparing actual counts $N_\Delta$ of constellations in the intervals of survival
$$\Delta H(p_k) \; = \; [p_k^2,\; p_{k+1}^2],$$
across several sampled intervals of survival, to the estimated populations from the relative population models
$w_{s,J}(\lambda)$.

We load the small primes to track $p_k$, and to lookup the $\lambda$ corresponding to $p_k$ we load the array <i>lambdaJ3E9[]</i>.
For our relative population models $w_{s,J}(\lambda)=w_{s,J}(p_k^\#)$ we start at $p_0=37$.  So $\lambda=1$ for $p_0=37$
and then
$$ \lambda_{J=3}(p_k) \; = \; \prod_{41}^{p_k} \frac{p-J-5}{p-J-4}$$

In [3]:
# we use primesE9 as the range for pk
# The interval of survival Delta-H(pk) goes from (pk)^2 to (p{k+1})^2
smallprimes = np.load('primesE9.npy')


In [4]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 3369.67 MB


In [5]:
lensmallprimes = len(smallprimes)
maxsmallprime = smallprimes[-1]
print(f"Length primes {lensmallprimes}  maxp {maxsmallprime}={maxsmallprime:.4e} maxhorizon {maxsmallprime**2} or {(maxsmallprime**2):.4e}")
# These values= primes 51961553  maxp 1023094327 maxhorizon 1046722001939582929

Length primes 51961884  maxp 1023101273=1.0231e+09 maxhorizon 1046736214814220529 or 1.0467e+18


In [6]:
# displaying the offset between smallprimes[] and lambdaE9[]
smallprimes[10:15]

array([37, 41, 43, 47, 53])

In [7]:
try:
    lambdaJ3E9 = np.load('lambdaJ3E9.npy')
except FileNotFoundError:
    i=10 # offset for p=37
    j=0
    lambdaJ3E9 = np.zeros(lensmallprimes-10)
    lambdaJ3E9[0] = 1
    while (i < (lensmallprimes-1)):
        j += 1
        i += 1
        lambdaJ3E9[j] = lambdaJ3E9[j-1] * (smallprimes[i]-5)/(smallprimes[i]-4)
    np.save('lambdaJ3E9.npy', lambdaJ3E9)

In [8]:
print(f"lenp {lensmallprimes} maxp {smallprimes[-1]} lenlam {len(lambdaJ3E9)} minlam {lambdaJ3E9[-1]}")

lenp 51961884 maxp 1023101273 lenlam 51961874 minlam 0.17773624700782842


In [9]:
# finding the index for a prime near the max for p^2 near 66 trillion
# for indexing, lambdaJ3E9[i] corresponds to smallprimes[i+10]
i=500000
while (smallprimes[i] < 8172000):
    i += 1
print(f"{i} p {smallprimes[i]} lambda {lambdaJ3E9[i-10]:.4f}")
i = len(smallprimes)-1
print(f"last {i} p {smallprimes[i]} lambda {lambdaJ3E9[i-10]:.4f}")
lambdaJ3E9[-100:-1]

550553 p 8172013 lambda 0.2317
last 51961883 p 1023101273 lambda 0.1777


array([0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773626, 0.17773626,
       0.17773626, 0.17773626, 0.17773626, 0.17773625, 0.17773625,
       0.17773625, 0.17773625, 0.17773625, 0.17773625, 0.17773625,
       0.17773625, 0.17773625, 0.17773625, 0.17773625, 0.17773625,
       0.17773625, 0.17773625, 0.17773625, 0.17773625, 0.17773625,
       0.17773625, 0.17773625, 0.17773625, 0.17773625, 0.17773

In [10]:
# calculate MertensC, the ratio between parameter lambdaJ3E9 and ln(pk)
MertensC0 = lambdaJ3E9[550541]*np.log(smallprimes[550551])
MertensC1 = lambdaJ3E9[-1]*np.log(smallprimes[-1])
MertensC0, MertensC1

(np.float64(3.687278218186093), np.float64(3.6873347209998624))

## Preparing for the comparison of sampled populations to estimates
The following code provides code for aggregating the populations of gaps from these samples, within intervals of survival $\Delta H(p)$

We save the data as a list for easy saving and loading.


In [11]:
# global variables describing the data for the figures
min_samp =3
max_samp = 101
len_samp = 1
num_DH = 0


In [12]:

# function for a light characterization of the data in a data-sample file of primes
def blockcheck(filename):
    global min_samp
    global max_samp
    global len_samp
    global num_DH

    testsample = np.load(filename, allow_pickle=True)
    min_samp = testsample[0]
    max_samp = testsample[-1]
    len_samp = len(testsample)

    i = 0
    minp = smallprimes[i]
    while (minp**2 < min_samp):
        i += 1
        minp = smallprimes[i]
    iminp = i
    minp = smallprimes[iminp]  # minp has the smallest p^2 above min_samp

    maxp = smallprimes[i]
    while (maxp**2 < max_samp):
        i += 1
        maxp = smallprimes[i]
    imaxp1 = i-1
    maxp1 = smallprimes[imaxp1] # maxp1 has the largest p^2 below max_samp

    # pick up marginal cases, where sample is a fragment of a DH
    if (imaxp1 <= iminp):  # sample is a fragment of a single DH
        # FRAGMENT - 
        # for now, terminate here.  This case is outside of the design for the interactive display
        num_DH = 0
        print(f"FRAGMENT: data sample is an incomplete fragment of $\Delta H$({minp})")
        return
        
    imaxp = imaxp1-1   # the start of the last interval [p^2,q^2] that fits inside the sample
    maxp = smallprimes[imaxp]

    # iminp, minp, imaxp, maxp, imaxp1, maxp1 are all associated with the array smallprimes[]
    # minp: the start of the first DH(p) in the sample
    # maxp: the start of the last DH(p) in the sample
    # maxp1: the end of the last DH(p) in the sample
    print(f"sample of {len_samp} primes from {min_samp} to {max_samp}")
    print(f"contains DeltaH from smallp[{iminp}]={minp} to smallp[{imaxp}]={maxp}")

    # the indices iDH0, iDH1, and iDH2 are associated with the array testsample[]
    iDH0 = 0
    while (testsample[iDH0] < minp**2):
        iDH0 += 1   # iDH0 marks the first prime beyond minp^2
    iDH1 = iDH0
    if (maxp > minp):
        while (testsample[iDH1] < maxp**2):
            iDH1 += 1       # iDH1 marks the first prime inside the last DH(p)
        iDH1 -= 1 
    iDH2 = iDH1
    while (testsample[iDH2] < (smallprimes[imaxp+1]**2)):
        iDH2 += 1           # iDH2 marks the last prime inside the last DH(p)
    iDH2 -= 1

    print(f"{imaxp-iminp+1} DeltaH cover sample[{iDH0}]={testsample[iDH0]} to sample[{iDH1}]={testsample[iDH1]} ending at sample[{iDH2}]={testsample[iDH2]}")
    print(f" vs minp^2 {minp**2} and maxp^2 {maxp**2} and {(smallprimes[imaxp+1]**2)}")
    


## Workflow for recomputing DHsEnn_xxx data files

In [105]:
# [31 May 2025] - these blocks contain approximately 50M primes each
# blockcheck('primeblockE18_1046twin.npy') # 1 DH around E18, at twin prime - this file contains over 130M primes
# blockcheck('primeblockE18_1046.npy') # fragment of one DH
# blockcheck('primeblockE17_200.npy')  # 2 DH
# blockcheck('primeblockE15_169A.npy')  # one DH
# blockcheck('primeblockE15_169B.npy')  # one DH
# blockcheck('primeblockE14_568A.npy')    # 1 DH
# blockcheck('primeblockE14_568B.npy')    # 2 DH
# blockcheck('primeblockE14_568C.npy')    # 3 DH
# blockcheck('primeblockE14_568D.npy')    # 3 DH
# blockcheck('primeblockE14_568E.npy')    # 7 DH
# blockcheck('primeblockE13_668.npy')    # 7 DH
blockcheck('primeblockE12_742A.npy')    # 10 DH
# blockcheck('primeblockE12_742B.npy')    # 18 DH
# blockcheck('primeblockE11_640.npy')    #  DH
# blockcheck('primeblockE10_108.npy')    # 506 DH
# blockcheck('primeblockE09_900.npy')    # 881 DH
# blockcheck('primeblockE09_360.npy')    # 1021 DH

sample of 30369351 primes from 7419909049811 to 7420809051199
contains DeltaH from smallp[198275]=2723951 to smallp[198284]=2724079
10 DeltaH cover sample[24]=7419909050453 to sample[23531637]=7420606398211 ending at sample[29045714]=7420769843849
 vs minp^2 7419909050401 and maxp^2 7420606398241 and 7420769843881


## Notes on samples of primes

| filename | num primes | $\min P$ | $\max P$ | $\# \Delta H$ | $\Delta H(p_0)$ | $\min P$ | $\Delta H(q_k)$ | $\max P$ |
| :--- | ---: | ---: | ---: | :---: | ---: | ---: | ---: | ---: |
| primeblockE09_360.npy | $55008596$ | $2400255059$ |$3600255031$ | $1021$ | $49003$ | $2401294057$ | $59981$ | $3599879993$ |
| primeblockE09_900.npy | $78902338$ | $7200254309$ |$9000254779$ | $881$ | $84857$ | $7200710471$ | $94847$ | $8996332747$ |
| primeblockE09_960.npy | $52355884$ | $8400254153$ |$9600254149$ | $557$ | $91673$ | $8403938941$ | $97967$ | $9598708711$ |
| primeblockE10_108.npy | $52072153$ | $9600254077$ | $10800254051$ | $506$ |$97987$ | $9601452193$|  $103913$ | $10799158553$ |
| primeblockE11_640.npy | $66214780$ | $6.400E11$ | $6.400E11$ | $94$ |$800011$ | $640017600137$|  $801103$ | $641772425447$ |
| primeblockE12_742.npy | $50612312$ | $7.421E12$ |$7.422E12$ | $18$ | $2724109$ | $7420769843911$ | $2724367$ | $7422230038127$ |
| primeblockE13_668.npy | $50267204$ | $6.679E13$ |$6.679E13$ | $7$ | $8172391$ | $66787974656917$ | $8172487$ | $66789543765101$ |
| primeblockE14_568.npy | $105954508$ | $5.677E14$ |$5.677E14$ | $7$ | $23826527$ | $567703388881771$ | $23826587$ | $567706819906781$ |
| primeblockE15_169.npy | $51337046$ | $1.694E15$ |$1.694E15$ | $1$ | $41161829$ | $1694296166625301$ | $41161829$ | $1694297813098769$ |
| primeblockE17_200.npy | $135549263$ | $2.00E17$ |$2.00E17$ | $2$ | $447216097$ | $200002237415913437$ | $447216101$ | $200002242782506531$ |
| primeblockE18_1046twin.npy | $130135262$ | $1.046E18$ | $1.046E18$ | $1$ |$1023094199$ | $1046721740027451601$|  $1023094201$ | $1046721744119828377$ |



## Accumulations over intervals $\Delta H(p)$
For comparison with the relative populations $w_{g,1}(p^\#)$, we accumulate the counts of gaps within the intervals of survival $\Delta H(p) = [p^2, q^2]$.

In [13]:
# checking sizes of arrays of data types.
exA1K = np.zeros(1000, dtype=int)
exB1K = np.zeros(1000, dtype=float)
exC1K = np.zeros(1000, dtype=bool)
exD1K = 'FBCD'*250
sys.getsizeof(exA1K),sys.getsizeof(exB1K),sys.getsizeof(exC1K),sys.getsizeof(exD1K) 

(8112, 8112, 1112, 1049)

In [14]:
# now the processing function to create the DH(p) data over the data sample from file
# createDH returns a list of 3 items:
#  [0]: samp0 = the smallest prime inside the first interval of survival for the data sample
#  [1]: [iminp, minp] = data for the first smallprime[] associated with the intervals of survival
#  [2]: DelH[numintervals,numconstellations] = the counts of constellations, indexed by the key-array, across the intervals DH(p)
def createDH(filename):
    global min_samp
    global max_samp
    global len_samp
    global num_DH

    testsample = np.load(filename, allow_pickle=True)
    min_samp = testsample[0]
    max_samp = testsample[-1]
    len_samp = len(testsample)

    i = 0
    minp = smallprimes[i]
    while (minp**2 < min_samp):
        i += 1
        minp = smallprimes[i]
    iminp = i
    minp = smallprimes[iminp]  # minp has the smallest p^2 above min_samp

    maxp = smallprimes[i]
    while (maxp**2 < max_samp):
        i += 1
        maxp = smallprimes[i]
    imaxp1 = i-1
    maxp1 = smallprimes[imaxp1] # maxp1 has the largest p^2 below max_samp

    # check for degenerate cases:  a single DH or a fragment of a DH
    if (imaxp1 <= iminp):  # sample is a fragment of a single DH
        # what to do what to do
        num_DH = 0
        # XXXXQHERE --- [11/15/25] RETURN with message -  the input doesn't fit the design....
        
    imaxp = imaxp1-1   # the start of the last interval [p^2,q^2] that fits inside the sample
    maxp = smallprimes[imaxp]

    # (iminp, minp), (imaxp, maxp), (imaxp1, maxp1) are all associated with the array smallprimes[]
    # minp: the smallprime (p for p^2) starting the first DH(p) in the sample
    # maxp: the smallprime (p for p^2) starting the last DH(p) in the sample
    # maxp1: the smallprime (q for q^2) ending the last DH(p) in the sample
    print(f"sample of {len_samp} primes from {min_samp} to {max_samp}")
    print(f"contains DeltaH from {iminp}:{minp} to {imaxp}:{maxp}")

    # the indices iDH0, iDH1, and iDH2 are associated with the input array testsample[]
    iDH0 = 0
    while (testsample[iDH0] < minp**2):
        iDH0 += 1   # iDH0 marks the first prime beyond minp^2
    iDH1 = iDH0
    if (maxp > minp):
        while (testsample[iDH1] < maxp**2):
            iDH1 += 1       # iDH1 marks the first prime inside the last DH(p)
        iDH1 -= 1 
    iDH2 = iDH1
    while (testsample[iDH2] < (smallprimes[imaxp+1]**2)):
        iDH2 += 1           # iDH2 marks the last prime inside the last DH(p)
    iDH2 -= 1

    # We have indices for the array of smallprimes[] and the array of testsample[]
    # Record the starting sample-prime and create the array of gaps
    samp0 = testsample[iDH0]
    if (iDH0 > 0):
        gapsample = testsample[iDH0:(iDH2+1)] - testsample[(iDH0-1):iDH2]
    # indexing across the arrays is gap[i] = sampleprimes[i+iDH0]-sampleprimes[i-1+iDH0]
    print(f"   gapsample {len(gapsample)} check {iDH2}-{iDH0}")

    # create the DelH 2-d arrays
    # for J=3 (columns indexed by J3keyarr)
    # and for J=5 (columns indexed by J5keyarr)
    numJ3 = len(J3keyarr)  # from a20s_class.py
    numJ5 = len(J5keyarr)
    numintervals = imaxp1 - iminp

    DelHs3 = np.zeros((numintervals, numJ3), dtype=int)
    DelHs5 = np.zeros((numintervals, numJ5), dtype=int)

    # We iterate through constellations s and intervals DH(p), counting occurrences
    # Length J=3 first ::::
    isj = 0
    while (isj < numJ3):
        current_key = J3keyarr[isj]
        current_s = J3dict[current_key] # for each constellation...
        print(f"\nNext constellation {current_s}\r", end='')
        
        ip = iminp  # index in smallprimes[]
        isam = 0  # index in gapsample[]

        idh = 0
        dhthresh = int(smallprimes[ip+1])**2
        
        # Match constellations - 
        # accumulate counts across gapsample
        while (isam <= (iDH2-iDH0-3) ):
            # does isam start a match for this constellation?
            if ((gapsample[isam] == current_s[0]) and (gapsample[isam+1] == current_s[1]) and (gapsample[isam+2] == current_s[2])):
                DelHs3[idh][isj] += 1

            isam += 1
            
            # did we cross into the next DH?
            if (testsample[iDH0+isam] > dhthresh): 
                ip += 1
                idh += 1
                dhthresh = int(smallprimes[ip+1])**2
                print(f"{current_s} Next DH {idh} prime {ip} {smallprimes[ip]} isample {isam} thresh {dhthresh} sample {testsample[iDH0+isam]}\r", end='')

        isj += 1

    # We iterate through constellations s and intervals DH(p), counting occurrences
    #  for Length J=5  ::::
    isj = 0
    while (isj < numJ5):
        current_key = J5keyarr[isj]
        current_s = J5dict[current_key] # for each constellation...
        print(f"\nNext constellation {current_s}\r",end='')
        
        ip = iminp  # index in smallprimes[]
        isam = 0  # index in gapsample[]

        idh = 0
        dhthresh = int(smallprimes[ip+1])**2
        
        # Match constellations - 
        # accumulate counts across gapsample
        while (isam <= (iDH2-iDH0-5) ):
            # does isam start a match for this constellation?
            if ((gapsample[isam] == current_s[0]) and (gapsample[isam+1] == current_s[1]) and (gapsample[isam+2] == current_s[2])
                 and (gapsample[isam+3] == current_s[3]) and (gapsample[isam+4] == current_s[4])):
                DelHs5[idh][isj] += 1

            isam += 1
            
            # did we cross into the next DH?
            if (testsample[iDH0+isam] > dhthresh): 
                ip += 1
                idh += 1
                dhthresh = int(smallprimes[ip+1])**2
                print(f"{current_s} Next DH {idh} prime {ip} {smallprimes[ip]} isample {isam} thresh {dhthresh} sample {testsample[iDH0+isam]}\r", end='')

        isj += 1

    DHlist = [samp0, [iminp, minp], DelHs3, DelHs5]
    return DHlist


In [142]:
DelHlist = createDH('primeblockE09_360.npy')
# DelHlist = createDH('primeblockE09_900.npy')
# DelHlist = createDH('primeblockE10_108.npy')
# DelHlist = createDH('primeblockE11_640.npy')
# DelHlist = createDH('primeblockE12_742A.npy')
# DelHlist = createDH('primeblockE12_742B.npy')
# DelHlist = createDH('primeblockE14_568A.npy')
# DelHlist = createDH('primeblockE14_568B.npy')
# DelHlist = createDH('primeblockE14_568C.npy')
# DelHlist = createDH('primeblockE14_568D.npy')
# DelHlist = createDH('primeblockE14_568E.npy')
# DelHlist = createDH('primeblockE15_169A.npy')
# DelHlist = createDH('primeblockE15_169B.npy')
# DelHlist = createDH('primeblockE17_200.npy')
# DelHlist = createDH('primeblockE18_1046twin.npy')


sample of 55008596 primes from 2400255059 to 3600255031
contains DeltaH from 5034:49003 to 6054:59981
   gapsample 54943608 check 54991567-47960

[2, 4, 2] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[4, 2, 6] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[6, 2, 4] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[2, 4, 12] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[12, 4, 2] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[6, 2, 6] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[6, 6, 2] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[2, 6, 6] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[2, 4, 8] Next DH 1020 prime 6054 59981 isample 54845521 thresh 3599880001 sample 3597720377
[8, 4, 2] Next 

In [143]:
DelHlist[0]

np.int64(2401294057)

In [144]:
DelHlist[1]

[5034, np.int64(49003)]

In [145]:
DHsample3 = DelHlist[2]
DHsample3.shape

(1021, 29)

In [146]:
DHsample3[:,0:8]

array([[ 9, 14, 12, ..., 15, 17, 19],
       [20,  8, 13, ..., 18, 21, 17],
       [28, 22, 23, ..., 25, 21, 18],
       ...,
       [37, 20, 20, ..., 45, 33, 33],
       [19, 20, 29, ..., 28, 23, 22],
       [31, 30, 39, ..., 43, 46, 44]], shape=(1021, 8))

In [147]:
DHsample5 = DelHlist[3]
DHsample5.shape

(1021, 18)

In [148]:
DHsample5[:,0:8]

array([[0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 1, 1, ..., 1, 0, 2],
       [0, 0, 0, ..., 0, 0, 3],
       [0, 0, 1, ..., 0, 1, 1]], shape=(1021, 8))

In [149]:
smallprimes[550579:550586]-smallprimes[550578:550585]

array([28, 24, 24,  2,  4,  8,  6])

## Saving and reading the tabled data
Sample code for saving the returned list to file and reading it to file.  We use pickle.dump() and pickle.load()

To keep the samples organized we use a naming convention 'DHExx_nnn' where xx is the exponent in scientific notation and nnn is the 
coefficient, three or more digits written without a decimal.

In [150]:
# File save - CAUTION - do not clobber existing data files
# USE WITH CARE ::: check that this filename agrees with data set above
with open('DHsE09_360',"wb") as fp:
    pickle.dump(DelHlist, fp)
fp.close()

In [108]:
# check read from pickled file
with open('DHsE14_568E',"rb") as fp2:
    DHff = pickle.load(fp2)
fp2.close()

In [109]:
DHff[0], DHff[1][0], DHff[1][1]


(np.int64(567703388881771), 1496935, np.int64(23826527))

In [110]:
DHff[2][:,0:6]

array([[ 313,  276,  273,  249,  258,  429],
       [ 625,  567,  531,  538,  493,  800],
       [1188, 1140, 1088, 1044, 1030, 1618],
       [1678, 1625, 1697, 1520, 1551, 2274],
       [4175, 3909, 3789, 3593, 3521, 5413],
       [ 903,  781,  875,  705,  758, 1202],
       [1772, 1642, 1637, 1619, 1516, 2346]])

In [111]:
DHff[2].shape, DHff[2].shape[0], DHff[2].shape[1]

((7, 29), 7, 29)

In [112]:
DHff[3].shape, DHff[3].shape[0], DHff[3].shape[1]

((7, 18), 7, 18)

### Lambda values at samples

| range label | iminp | minp | $\lambda(minp)$ | $\lambda_{J=3}(minp)$ | $P_0$ in $\Delta H(minp)$ |
| :--- | ---: | ---: | :---: | :---: | ---: |
| E09_360 | $5034$ | $49003$ | $0.3455065$ | $0.3413388$ | $2401294057$ |
| E09_900 | $8264$ | $84857$ | $0.3287857$ | $0.3248191$ | $7200710471$ |
| E10_108 | $9416$ | $97987$ | $0.3246621$ | $0.3207451$ | $9601452193$ |
| E11_640 | $63950$ | $800011$ | $0.2745705$ | $0.2712576$ | $640017600137$ |
| E12_742 | $198285$ | $2724109$ | $0.2518756$ | $0.2488364$ | $7420769843911$ |
| E14_568 | $1496935$ | $23826527$ | $0.2197262$ | $0.2170749$ | $567703388881771$ |
| E15_169 | $2500003$ | $41161829$ | $0.2128746$ | $0.2103060$ | $1694296166625301$ |
| E17_200 | $23713133$ | $447216097$ | $0.1873815$ | $0.1851205$ | $200002237415913437$ |
| E18_1046 | $51961545$ | $1023094199$ | $0.1799071$ | $0.1777363$ | $1046721740027451601$ |

## Code to combine adjacent data files DHsExx_nnn

As our computations continue, we combine existing DHsE-files to get the statistics for more intervals of survival

In [188]:
# function for combining two DH files
def combineDH(filename1, filename2):
    with open(filename1,"rb") as fp1:
        DHf1 = pickle.load(fp1)
    fp1.close()

    with open(filename2,"rb") as fp2:
        DHf2 = pickle.load(fp2)
    fp2.close()

    # extract the data fields from the imported DH
    P0_1 = DHf1[0]
    iminp_1 = DHf1[1][0]
    minp_1 = DHf1[1][1]
    nDHJ3_1 = DHf1[2].shape[0]
    jDHJ3_1 = DHf1[2].shape[1]
    nDHJ5_1 = DHf1[3].shape[0]
    jDHJ5_1 = DHf1[3].shape[1]

    P0_2 = DHf2[0]
    iminp_2 = DHf2[1][0]
    minp_2 = DHf2[1][1]
    nDHJ3_2 = DHf2[2].shape[0]
    jDHJ3_2 = DHf2[2].shape[1]
    nDHJ5_2 = DHf2[3].shape[0]
    jDHJ5_2 = DHf2[3].shape[1]

    print(f"Trying to extend {iminp_1}-{iminp_1 + nDHJ3_1} by {iminp_2}-{iminp_2 + nDHJ3_2}")
    
    # check that these runs of intervals can be combined
    if iminp_1 > iminp_2:
        # blocks should be called in reverse order
        print(f"Error: blocks may be reversed {iminp_1} vs {iminp_2}")
        return None
        
    if iminp_2 > (iminp_1 + nDHJ3_1):
        # there is a gap between these blocks
        print(f"Error: blocks are not contiguous {iminp_1} + {nDHJ3_1} vs {iminp_2}")
        return None

    if (iminp_2 + nDHJ3_2) < (iminp_1 + nDHJ3_1):
        # the second block is covered by the first block
        print(f"Error: block {iminp_2} - {iminp_2+nDHJ3_2} is covered by {iminp_1} - {iminp_1 + nDHJ3_1}")
        return None

    # We append the second DH block to the first one and record the result
    iminp = iminp_1
    minp = smallprimes[iminp]
    ibndp = iminp + nDHJ3_1
    offset = ibndp - iminp_2  # offset marks the first entry in DH_2 that extends DH_1

    if (offset < 0) or (offset >= nDHJ3_2):
        printf(f"Unseen Error: offset {offset}")
        return None

    jp = iminp_2 + offset  # starting point in second DH block
    jmaxJ3 = jDHJ3_1 if (jDHJ3_1 > jDHJ3_2) else jDHJ3_2
    nDH = nDHJ3_1 + nDHJ3_2 - offset

    DHJ3ff = np.zeros((nDH, jmaxJ3), dtype=int)
    DHJ3ff[0:nDHJ3_1, 0:jDHJ3_1] = DHf1[2]
    DHJ3ff[nDHJ3_1:nDH, 0:jDHJ3_2] = DHf2[2][offset:nDHJ3_2, 0:jDHJ3_2]
    
    print(f"Combined shape {DHJ3ff.shape} Total population {np.sum(DHJ3ff)} ")

    jmaxJ5 = jDHJ5_1 if (jDHJ5_1 > jDHJ5_2) else jDHJ5_2
    DHJ5ff = np.zeros((nDH, jmaxJ5), dtype=int)
    DHJ5ff[0:nDHJ5_1, 0:jDHJ5_1] = DHf1[3]
    DHJ5ff[nDHJ5_1:nDH, 0:jDHJ5_2] = DHf2[3][offset:nDHJ5_2, 0:jDHJ5_2]
    
    print(f"Combined shape {DHJ5ff.shape} Total population {np.sum(DHJ5ff)}")

    DHret = [P0_1, (iminp, minp), DHJ3ff, DHJ5ff]

    return DHret

    

In [189]:
DHs15 = combineDH('DHsE15_169A', 'DHsE15_169B')

Trying to extend 2500003-2500004 by 2500004-2500005
Combined shape (2, 29) Total population 301245 
Combined shape (2, 18) Total population 1392


In [190]:
DHs15[0], DHs15[1]

(np.int64(1694296166625301), (2500003, np.int64(41161829)))

In [191]:
DHs15[2].shape, DHs15[3].shape

((2, 29), (2, 18))

In [192]:
with open('DHsE15_169',"wb") as fp:
    pickle.dump(DHs15, fp)
fp.close()

### combining the data blocks for E14_568 A-E.

In [193]:
DHs14 = combineDH('DHsE14_568A', 'DHsE14_568B')

Trying to extend 1496928-1496929 by 1496929-1496931
Combined shape (3, 29) Total population 378729 
Combined shape (3, 18) Total population 1820


In [194]:
with open('DHsE14_568',"wb") as fp:
    pickle.dump(DHs14, fp)
fp.close()

In [195]:
DHs14 = combineDH('DHsE14_568', 'DHsE14_568C')

Trying to extend 1496928-1496931 by 1496931-1496934
Combined shape (6, 29) Total population 613737 
Combined shape (6, 18) Total population 3005


In [196]:
with open('DHsE14_568',"wb") as fp:
    pickle.dump(DHs14, fp)
fp.close()

In [197]:
DHs14 = combineDH('DHsE14_568', 'DHsE14_568D')

Trying to extend 1496928-1496934 by 1496934-1496937
Combined shape (9, 29) Total population 849814 
Combined shape (9, 18) Total population 4140


In [198]:
with open('DHsE14_568',"wb") as fp:
    pickle.dump(DHs14, fp)
fp.close()

In [199]:
DHs14 = combineDH('DHsE14_568', 'DHsE14_568E')

Trying to extend 1496928-1496937 by 1496935-1496942
Combined shape (14, 29) Total population 1280988 
Combined shape (14, 18) Total population 6275


In [200]:
with open('DHsE14_568',"wb") as fp:
    pickle.dump(DHs14, fp)
fp.close()

In [201]:
DHs14[3]

array([[13, 34, 26, 34, 34, 27, 30, 34, 24, 45, 35, 44, 42, 47, 40, 39,
        17,  1],
       [22, 51, 37, 43, 52, 60, 49, 55, 37, 73, 55, 66, 59, 59, 76, 58,
        30,  2],
       [ 3, 29, 19, 18, 32, 22,  5, 18, 12, 26, 37, 31, 20, 22, 34, 23,
        14,  5],
       [ 3,  7,  1,  2,  6,  4,  3,  1,  2,  8,  4,  5,  3,  4,  6,  2,
         3,  0],
       [20, 59, 53, 41, 65, 47, 50, 47, 48, 74, 85, 76, 65, 67, 64, 78,
        42,  4],
       [ 1,  4,  6,  5,  6,  6,  9,  8,  5, 12, 11, 15, 14, 11,  9,  9,
         5,  0],
       [15, 66, 48, 46, 68, 56, 48, 49, 49, 63, 79, 74, 64, 59, 56, 80,
        28,  0],
       [ 0,  3,  2,  3,  8,  4,  5,  2,  1,  4,  9,  6,  3,  1,  4,  7,
         4,  2],
       [ 2,  9,  4,  6,  8,  8,  6,  5, 11,  9, 12,  4,  7,  3,  7,  9,
         8,  1],
       [ 2, 18, 14, 12, 13, 10, 17, 17, 10, 25, 17, 15, 19, 18, 16, 25,
        12,  2],
       [ 3, 24, 21, 16, 26, 15, 22, 20, 22, 27, 32, 20, 32, 30, 30, 31,
        15,  1],
       [14, 61, 46, 5

In [ ]:
# combine E12_742 A-B

In [202]:
DHs12 = combineDH('DHsE12_742A', 'DHsE12_742B')

Trying to extend 198275-198285 by 198285-198303
Combined shape (28, 29) Total population 532478 
Combined shape (28, 18) Total population 3286


In [203]:
with open('DHsE12_742',"wb") as fp:
    pickle.dump(DHs12, fp)
fp.close()